# IMPORTS

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import chi2
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
import joblib

# DATA LOADING

In [ ]:
data = pd.read_csv('data_up.csv')
data.head(5)

,SEQN,Gender,Age,Annual-Family-Income,Ratio-Family-Income-Poverty,X60-sec-pulse,Systolic,Diastolic,Weight,Height,...,Total-Cholesterol,HDL,Glycohemoglobin,Vigorous-work,Moderate-work,Health-Insurance,Diabetes,Blood-Rel-Diabetes,Blood-Rel-Stroke,CoronaryHeartDisease
0,2,1,77,8,5.00,68,98,56,75.4,174.0,...,5.56,1.39,4.7,3,3,1,2,2,2,0
1,5,1,49,11,5.00,66,122,83,92.5,178.3,...,7.21,1.08,5.5,1,1,1,2,2,2,0
2,12,1,37,11,4.93,64,174,99,99.2,180.0,...,4.03,0.98,5.2,2,1,1,2,1,1,0
3,13,1,70,3,1.07,102,130,66,63.6,157.7,...,8.12,1.28,7.6,3,3,1,1,1,2,0
4,14,1,81,5,2.67,72,136,61,75.5,166.2,...,4.50,1.04,5.8,1,1,1,2,2,2,0


# FEATURE ENGINEERING

In [ ]:
X = data.drop(columns=['SEQN'])
y = data['CoronaryHeartDisease']

In [ ]:
X_cat = MinMaxScaler().fit_transform(X)
chi_scores, p_values = chi2(X_cat, y)
chi2_results = pd.Series(chi_scores, index=X.columns).sort_values(ascending=False)
print(chi2_results.head(25))
X_selected = X[chi2_results.head(25).index]
X_selected = X_selected.drop(columns=['CoronaryHeartDisease'])

CoronaryHeartDisease           35571.000000
Age                              316.913617
Gender                           110.950323
Diabetes                          31.214014
Health-Insurance                  20.662408
Blood-Rel-Stroke                  17.907611
Vigorous-work                     14.123127
Annual-Family-Income               9.570904
Uric.Acid                          8.627958
Blood-Rel-Diabetes                 8.592963
Glucose                            7.846286
Creatinine                         7.837271
Total-Cholesterol                  6.898363
Glycohemoglobin                    6.823632
Lymphocyte                         6.340687
Platelet-count                     5.013676
Cholesterol                        4.853257
Moderate-work                      4.162391
Red-Cell-Distribution-Width        3.666581
X60-sec-pulse                      3.663029
HDL                                3.507004
Diastolic                          2.670472
Systolic                        

# ML Model

In [ ]:
models = {
    'RandomForestClassifier' : RandomForestClassifier(random_state=42),
    'LogisticRegression' : LogisticRegression(random_state=42),
    'SupportVectorMachine' : SVC(random_state=42),
    'K-NearestNeighbors' : KNeighborsClassifier(),
    'DecisionTree' : DecisionTreeClassifier(random_state=42),
    'GaussianNaiveBayes' : GaussianNB(),
}

In [ ]:

scaler = StandardScaler()
for name, model in models.items():
    X_selected_scaled = scaler.fit_transform(X_selected)
    X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy: {accuracy}")

RandomForestClassifier Accuracy: 0.9581984897518878


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression Accuracy: 0.9576591154261057
SupportVectorMachine Accuracy: 0.9583333333333334
K-NearestNeighbors Accuracy: 0.9550970873786407
DecisionTree Accuracy: 0.9217907227615966
GaussianNaiveBayes Accuracy: 0.8959007551240561


# DL MODEL - MLP

In [ ]:
scaler = StandardScaler()
X_selected_scaled = scaler.fit_transform(X_selected)
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)
model_mlp = keras.Sequential([
    keras.layers.Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),

    keras.layers.Dense(128, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),

    keras.layers.Dense(64, activation='relu'),
    keras.layers.BatchNormalization(),

    keras.layers.Dense(1, activation='sigmoid')
])
opt = keras.optimizers.Adam(learning_rate=0.0005)
model_mlp.compile(optimizer=opt,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)
history = model_mlp.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


742/742 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.7892 - loss: 0.4764 - val_accuracy: 0.9587 - val_loss: 0.1567
Epoch 2/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9588 - loss: 0.1567 - val_accuracy: 0.9592 - val_loss: 0.1526
Epoch 3/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9587 - loss: 0.1475 - val_accuracy: 0.9589 - val_loss: 0.1442
Epoch 4/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9584 - loss: 0.1426 - val_accuracy: 0.9590 - val_loss: 0.1416
Epoch 5/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9595 - loss: 0.1387 - val_accuracy: 0.9589 - val_loss: 0.1452
Epoch 6/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9581 - loss: 0.1439 - val_accuracy: 0.9592 - val_loss: 0.1395
Epoch 7/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9572 - loss: 0.1410 - val_accuracy: 0.9594 - val_loss: 0.1359
Epoch 8/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9570 - loss: 0.1441 - val_accuracy: 0.9592 - val_

In [ ]:
test_loss, test_acc = model_mlp.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {test_acc:.4f}")

232/232 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9566 - loss: 0.1354

Test Accuracy: 0.9581


# Pipeline

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', model_mlp)
])
pipeline.fit(X_train, y_train)
joblib.dump(model, "heart_disease_model.joblib")

927/927 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9612 - loss: 0.1388


['heart_disease_model.joblib']

In [ ]:
joblib.load("heart_disease_model.joblib")
new_data = pd.DataFrame([{
    'Age': 56,
    'Gender': 1,
    'Diabetes': 0,
    'Health-Insurance': 1,
    'Blood-Rel-Stroke': 0,
    'Vigorous-work': 1,
    'Annual-Family-Income': 50000,
    'Uric.Acid': 6.2,
    'Blood-Rel-Diabetes': 1,
    'Glucose': 120,
    'Creatinine': 1.0,
    'Total-Cholesterol': 190,
    'Glycohemoglobin': 5.8,
    'Lymphocyte': 2.1,
    'Platelet-count': 230,
    'Cholesterol': 180,
    'Moderate-work': 0,
    'Red-Cell-Distribution-Width': 12.5,
    'X60-sec-pulse': 78,
    'HDL': 45,
    'Diastolic': 85,
    'Systolic': 140,
    'Monocyte': 0.3,
    'Eosinophils': 0.2
}])
prediction = model.predict(new_data)[0]
print("Heart Disease Risk (0=No, 1=Yes):", prediction)

Heart Disease Risk (0=No, 1=Yes): 0
